# 03 — Il confronto

Tutti i modelli finiti (feature, baseline, transformer con i loro seed), sugli stessi fold **per gruppo**. Le
metriche principali, fissate prima dei risultati: **F1 macro a 5 classi** e **accuratezza binaria**, per
testata. Si riportano anche accuratezza a 5 classi, tolerant 0.5 e 1, e il dettaglio per livello.

Gli intervalli al 95% vengono da un **bootstrap per gruppo** (2000 ricampionamenti dei gruppi di testate: le
testate dello stesso gruppo sbagliano insieme). Per ogni transformer, la **differenza** con il classificatore a
feature sugli stessi ricampionamenti.

Per le configurazioni con più seed c'è anche l'**ensemble** (`..._ensembleN`): la media delle probabilità dei seed
per ogni articolo, decisa prima di vedere i risultati.

Gira da solo alla fine di ogni esperimento della coda: si può guardare anche mentre la coda lavora.

In [1]:
import json
import re
from pathlib import Path

import numpy as np
import pandas as pd

from comune import CLASSI, RESULTS, VERSIONI, metriche_di, per_testata, per_testata_media

SCHEMA = "fold_gruppo"
B = 2000

In [2]:
modelli = {}
for f in sorted(RESULTS.glob(f"feature*_{SCHEMA}_articoli.csv")):
    modelli[f.name.replace(f"_{SCHEMA}_articoli.csv", "")] = pd.read_csv(f)
for d in sorted((RESULTS / "transformer").glob("*")):
    if (d / "FATTO").exists():
        modelli[d.name] = pd.read_csv(d / "articoli.csv")
# ensemble dei seed: per ogni configurazione con più seed finiti, la media delle probabilità per articolo
per_config = {}
for nome in list(modelli):
    m = re.match(r"(.+)_s\d+$", nome)
    if m:
        per_config.setdefault(m[1], []).append(nome)
for config, nomi_seed in per_config.items():
    if len(nomi_seed) > 1:
        parti = [modelli[n].set_index("article_id") for n in nomi_seed]
        e = parti[0][["domain", "fold", "y_true"]].copy()
        e[CLASSI] = sum(p[CLASSI] for p in parti) / len(parti)
        e["y_pred"] = e[CLASSI].to_numpy().argmax(1)
        modelli[f"{config}_ensemble{len(nomi_seed)}"] = e.reset_index()
print("modelli finiti:", list(modelli))
gruppo = pd.read_csv(VERSIONI / "fold_finale.csv").set_index("domain").gruppo

modelli finiti: ['feature', 'modernbert_base_1024_s42']


In [3]:
PRINCIPALI = ["f1_macro", "bin_accuratezza", "accuratezza", "tolerant_0.5", "tolerant_1", "bin_f1_macro"]
righe, testate = {}, {}
for nome, p in modelli.items():
    for agg, fun in [("voto", per_testata), ("media", per_testata_media)]:
        t = fun(p); testate[(nome, agg)] = t
        righe[(nome, agg)] = {k: v for k, v in metriche_di(t.y_true, t.y_pred).items() if not isinstance(v, list)}
for f in sorted(RESULTS.glob(f"baseline_*_{SCHEMA}_metriche.json")):
    righe[(f.name.split(f"_{SCHEMA}")[0], "-")] = {k: v for k, v in json.load(open(f))["testata_voto"].items() if not isinstance(v, list)}
tutto = pd.DataFrame(righe).T
tutto.index.names = ["modello", "aggregazione"]
tutto.to_csv(RESULTS / "03_confronto_tutte_le_metriche.csv")
tutto[PRINCIPALI].round(3)

f1_macro  bin_accuratezza  accuratezza  \
modello                  aggregazione                                           
feature                  voto             0.282            0.677        0.357   
                         media            0.265            0.673        0.342   
modernbert_base_1024_s42 voto             0.213            0.599        0.238   
                         media            0.244            0.632        0.279   
baseline_a_caso          -                0.176            0.528        0.212   
baseline_maggioranza     -                0.104            0.643        0.353   

                                       tolerant_0.5  tolerant_1  bin_f1_macro  
modello                  aggregazione                                          
feature                  voto                 0.556       0.755         0.657  
                         media                0.548       0.755         0.648  
modernbert_base_1024_s42 voto                 0.439       0.639         0.584  
                         media                0.465       0.651         0.613  
baseline_a_caso          -                    0.383       0.554         0.489  
baseline_maggioranza     -                    0.498       0.643         0.391

## Seed: media e deviazione standard

Gli esperimenti con lo stesso nome a parte il seed (`..._s42`, `..._s1`, `..._s2`).

In [4]:
x = tutto[PRINCIPALI].reset_index()
x["config"] = x.modello.str.replace(r"_s\d+$", "", regex=True)
seed = x.groupby(["config", "aggregazione"])[PRINCIPALI].agg(["mean", "std", "count"])
seed.to_csv(RESULTS / "03_confronto_seed.csv")
seed.round(3)

f1_macro           bin_accuratezza      \
                                      mean std count            mean std   
config               aggregazione                                          
baseline_a_caso      -               0.176 NaN     1           0.528 NaN   
baseline_maggioranza -               0.104 NaN     1           0.643 NaN   
feature              media           0.265 NaN     1           0.673 NaN   
                     voto            0.282 NaN     1           0.677 NaN   
modernbert_base_1024 media           0.244 NaN     1           0.632 NaN   
                     voto            0.213 NaN     1           0.599 NaN   

                                        accuratezza           tolerant_0.5  \
                                  count        mean std count         mean   
config               aggregazione                                            
baseline_a_caso      -                1       0.212 NaN     1        0.383   
baseline_maggioranza -                1       0.353 NaN     1        0.498   
feature              media            1       0.342 NaN     1        0.548   
                     voto             1       0.357 NaN     1        0.556   
modernbert_base_1024 media            1       0.279 NaN     1        0.465   
                     voto             1       0.238 NaN     1        0.439   

                                            tolerant_1           bin_f1_macro  \
                                  std count       mean std count         mean   
config               aggregazione                                               
baseline_a_caso      -            NaN     1      0.554 NaN     1        0.489   
baseline_maggioranza -            NaN     1      0.643 NaN     1        0.391   
feature              media        NaN     1      0.755 NaN     1        0.648   
                     voto         NaN     1      0.755 NaN     1        0.657   
modernbert_base_1024 media        NaN     1      0.651 NaN     1        0.613   
                     voto         NaN     1      0.639 NaN     1        0.584   

                                             
                                  std count  
config               aggregazione            
baseline_a_caso      -            NaN     1  
baseline_maggioranza -            NaN     1  
feature              media        NaN     1  
                     voto         NaN     1  
modernbert_base_1024 media        NaN     1  
                     voto         NaN     1

## Intervalli e differenze con il feature (bootstrap per gruppo)

In [5]:
rif = ("feature", "voto")
nomi = [k for k in testate if k[1] == "media" or k == rif]
dom = testate[rif].index
g = gruppo.reindex(dom).to_numpy()
gruppi = np.unique(g)
pos = {x: np.where(g == x)[0] for x in gruppi}
rng = np.random.default_rng(0)
campioni = [np.concatenate([pos[x] for x in rng.choice(gruppi, len(gruppi))]) for _ in range(B)]


def due(t, idx):
    m = metriche_di(t.y_true.to_numpy()[idx], t.y_pred.to_numpy()[idx])
    return m["f1_macro"], m["bin_accuratezza"]


boot = {k: np.array([due(testate[k].reindex(dom), idx) for idx in campioni]) for k in nomi}
righe = {}
for k in nomi:
    b = boot[k]
    r = {"F1 macro": tutto.loc[k, "f1_macro"], "F1 2.5%": np.percentile(b[:, 0], 2.5), "F1 97.5%": np.percentile(b[:, 0], 97.5),
         "binario": tutto.loc[k, "bin_accuratezza"], "bin 2.5%": np.percentile(b[:, 1], 2.5), "bin 97.5%": np.percentile(b[:, 1], 97.5)}
    if k != rif:
        d = b - boot[rif]
        r.update({"ΔF1 vs feature": tutto.loc[k, "f1_macro"] - tutto.loc[rif, "f1_macro"], "ΔF1 2.5%": np.percentile(d[:, 0], 2.5),
                  "ΔF1 97.5%": np.percentile(d[:, 0], 97.5), "P(ΔF1>0)": (d[:, 0] > 0).mean(),
                  "Δbin vs feature": tutto.loc[k, "bin_accuratezza"] - tutto.loc[rif, "bin_accuratezza"],
                  "Δbin 2.5%": np.percentile(d[:, 1], 2.5), "Δbin 97.5%": np.percentile(d[:, 1], 97.5), "P(Δbin>0)": (d[:, 1] > 0).mean()})
    righe[f"{k[0]} ({k[1]})"] = r
intervalli = pd.DataFrame(righe).T
intervalli.to_csv(RESULTS / "03_confronto_intervalli.csv")
intervalli.round(3)

,F1 macro,F1 2.5%,F1 97.5%,binario,bin 2.5%,bin 97.5%,ΔF1 vs feature,ΔF1 2.5%,ΔF1 97.5%,P(ΔF1>0),Δbin vs feature,Δbin 2.5%,Δbin 97.5%,P(Δbin>0)
feature (voto),0.282,0.215,0.345,0.677,0.608,0.743,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
feature (media),0.265,0.195,0.329,0.673,0.603,0.741,-0.017,-0.051,0.015,0.158,-0.004,-0.035,0.026,0.366
modernbert_base_1024_s42 (media),0.244,0.189,0.300,0.632,0.554,0.709,-0.038,-0.123,0.051,0.216,-0.045,-0.115,0.025,0.096


## Per livello e matrici di confusione (testata, media delle probabilità; feature col voto)

In [6]:
LIV = ["l1", "l2", "l3", "l4", "l5"]
per_livello = pd.DataFrame({f"{k[0]} ({k[1]})": {f"{l} {q}": tutto.loc[k, f"{l}_{q}"] for l in LIV for q in ["recall", "f1"]} for k in nomi}).T
display(per_livello.round(2))
for k in nomi:
    t = testate[k]
    print(f"\n{k[0]} ({k[1]})")
    print(pd.crosstab(t.y_true.map(dict(enumerate(LIV))), t.y_pred.map(dict(enumerate(LIV))), rownames=["vero"], colnames=["predetto"]).to_string())

,l1 recall,l1 f1,l2 recall,l2 f1,l3 recall,l3 f1,l4 recall,l4 f1,l5 recall,l5 f1
feature (voto),0.21,0.24,0.44,0.38,0.14,0.15,0.55,0.52,0.10,0.12
feature (media),0.21,0.25,0.40,0.35,0.08,0.09,0.56,0.51,0.10,0.12
modernbert_base_1024_s42 (media),0.18,0.18,0.38,0.34,0.05,0.05,0.32,0.40,0.32,0.25



feature (voto)
predetto  l1  l2  l3  l4  l5
vero                        
l1         7  14   2   8   2
l2        10  28   9  12   4
l3         4  15   5  10   3
l4         4  14  15  52  10
l5         1  12   0  24   4

feature (media)
predetto  l1  l2  l3  l4  l5
vero                        
l1         7  13   2   9   2
l2        10  25   9  13   6
l3         4  15   3  13   2
l4         2  12  17  53  11
l5         1  13   0  23   4

modernbert_base_1024_s42 (media)
predetto  l1  l2  l3  l4  l5
vero                        
l1         6  12  10   2   3
l2        13  24   7   5  14
l3         8  13   2   6   8
l4         6  22  12  30  25
l5         0   9   7  12  13
